# Honeydew AI — расширенное облачное обучение

Цель — не менее 200 000 обучающих примеров: 10 000 оригинальных синтетических композиций + до 190 000 MIDI из PDMX (только Public Domain/CC0 без конфликта лицензии и с доступным MIDI). Скрипт остановится, если не удастся собрать все 190 000 внешних файлов: недостающие композиции не подменяются дубликатами.

**Всё хранится на Google Drive:** загрузка PDMX продолжает `.part`-файлы, распаковка пропускает уже обработанные MIDI, сборка датасета продолжает `.inprogress`, а модель сохраняет атомарный checkpoint каждые 25 шагов и после каждой эпохи. Если Colab отключится, снова открой эту тетрадь, выполни ячейки сверху вниз и запусти обучение с `--resume`.


In [ ]:
!nvidia-smi || true
import torch
print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())


In [ ]:
!if [ ! -d /content/honeydew/.git ]; then git clone --depth 1 --branch ai-beatmaker https://github.com/djdhsvsbdb-design/honeydew.git /content/honeydew; fi
%cd /content/honeydew
!python -m pip install -r tools/ai/requirements.txt


In [ ]:
from google.colab import drive
from pathlib import Path
import json
drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/Honeydew')
ROOT.mkdir(parents=True, exist_ok=True)
PDMX_DIR = ROOT / 'pdmx'
DATA_DIR = ROOT / 'data'
CHECKPOINT_DIR = ROOT / 'checkpoints'
for folder in (PDMX_DIR, DATA_DIR, CHECKPOINT_DIR):
    folder.mkdir(parents=True, exist_ok=True)
CHECKPOINT = CHECKPOINT_DIR / 'honeydew-music.pt'
DATASET = DATA_DIR / 'honeydew-200k.jsonl'
STATUS = ROOT / 'pipeline-status.json'
def mark_stage(stage, details=None):
    payload = {'stage': stage, 'details': details or {}}
    STATUS.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding='utf-8')
    print('Pipeline status saved:', payload)
print('Drive storage ready:', ROOT)
print('Checkpoint:', CHECKPOINT)
print('Dataset:', DATASET)


In [ ]:
# Все промежуточные файлы лежат на Google Drive; скрипты умеют продолжать после прерывания.
import subprocess, sys
subprocess.run([sys.executable, 'tools/ai/download_pdmx.py', '--out-dir', str(PDMX_DIR), '--limit', '190000'], check=True)
mark_stage('pdmx_downloaded', {'pdmx_dir': str(PDMX_DIR)})
subprocess.run([sys.executable, 'tools/ai/build_dataset.py', '--synthetic-songs', '10000', '--external-limit', '190000', '--external-manifest', str(PDMX_DIR / 'external-manifest.jsonl'), '--out', str(DATASET)], check=True)
mark_stage('dataset_ready', {'dataset': str(DATASET), 'lines': sum(1 for _ in DATASET.open(encoding='utf-8'))})
print('Dataset rows:', sum(1 for _ in DATASET.open(encoding='utf-8')))


In [ ]:
# Checkpoint сохраняется каждые 25 шагов и в конце каждой эпохи прямо на Google Drive.
# При перезапуске ячейки --resume продолжит с сохранённого шага текущей эпохи.
import subprocess, sys
subprocess.run([sys.executable, 'tools/ai/train_music.py', '--data', str(DATASET), '--epochs', '20', '--batch-size', '8', '--block', '256', '--val-fraction', '0.1', '--save-every-steps', '25', '--out', str(CHECKPOINT), '--resume'], check=True)
mark_stage('training_finished', {'checkpoint': str(CHECKPOINT)})


In [ ]:
!python tools/ai/generate_music.py --checkpoint /content/drive/MyDrive/Honeydew/checkpoints/honeydew-music.best.pt --style trap --bars 8 --bpm 145 --out /content/drive/MyDrive/Honeydew/generated/honeydew-generated


In [ ]:
from google.colab import files
from pathlib import Path

for filename in [
    '/content/drive/MyDrive/Honeydew/checkpoints/honeydew-music.pt',
    '/content/drive/MyDrive/Honeydew/checkpoints/honeydew-music.best.pt',
    '/content/drive/MyDrive/Honeydew/generated/honeydew-generated.mid',
    '/content/drive/MyDrive/Honeydew/generated/honeydew-generated.json',
]:
    if Path(filename).exists():
        files.download(filename)
    else:
        print(f'Пока нет файла: {filename}')
